In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import re
import string
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score


- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development.

In [4]:
# Read the labeled training data. The first existing path is used so the notebook
# works both in the Ironhack repository structure and with the uploaded files.
train_candidates = [
    Path("data/kg_train.csv"),
    Path("kg_train.csv"),
    Path("kg_train(2).csv"),
]
train_path = next((p for p in train_candidates if p.exists()), None)
if train_path is None:
    raise FileNotFoundError("Could not find kg_train.csv")

data = pd.read_csv(train_path, encoding="latin-1")
data = data[["text", "label"]].copy()
data["text"] = data["text"].fillna("").astype(str)
data["label"] = data["label"].astype(int)

print("Dataset shape:", data.shape)
print("Class counts:")
print(data["label"].value_counts().sort_index().rename(index={0: "ham", 1: "spam"}))
data.head()


Dataset shape: (5964, 2)
Class counts:
label
ham     3386
spam    2578
Name: count, dtype: int64


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0


### Let's divide the training and test set into two partitions

In [5]:
# Stratified split keeps approximately the same HAM/SPAM proportion in both sets.
data_train, data_val = train_test_split(
    data,
    test_size=0.20,
    random_state=42,
    stratify=data["label"],
)

data_train = data_train.copy()
data_val = data_val.copy()

print("Training shape:", data_train.shape)
print("Validation shape:", data_val.shape)
print("Training label distribution:")
print(data_train["label"].value_counts(normalize=True).sort_index())
print("Validation label distribution:")
print(data_val["label"].value_counts(normalize=True).sort_index())


Training shape: (4771, 2)
Validation shape: (1193, 2)
Training label distribution:
label
0    0.567805
1    0.432195
Name: proportion, dtype: float64
Validation label distribution:
label
0    0.567477
1    0.432523
Name: proportion, dtype: float64


## Data Preprocessing

In [6]:
print("Punctuation:", string.punctuation)
print("Example stopwords:", sorted(list(ENGLISH_STOP_WORDS))[100:110])

# SnowballStemmer does not require downloaded NLTK corpora and is used only as an
# offline fallback if the WordNet lemmatizer data are unavailable.
from nltk.stem.snowball import SnowballStemmer
from nltk.stem import WordNetLemmatizer
snowball = SnowballStemmer("english")
wordnet_lemmatizer = WordNetLemmatizer()


Punctuation: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
Example stopwords: ['formerly', 'forty', 'found', 'four', 'from', 'front', 'full', 'further', 'get', 'give']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [7]:
def remove_html(text):
    """Remove inline JavaScript/CSS, HTML comments, and remaining HTML tags."""
    text = re.sub(r"<script.*?>.*?</script>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<style.*?>.*?</style>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
    text = re.sub(r"<[^>]+>", " ", text)
    return text

for frame in (data_train, data_val):
    frame["preprocessed_text"] = frame["text"].apply(remove_html)

data_train[["text", "preprocessed_text"]].head()


,text,preprocessed_text
1416,"DEAR FRIEND,PLEASE, REPLY TO MY PRIVATE E-MAIL...","DEAR FRIEND,PLEASE, REPLY TO MY PRIVATE E-MAIL..."
4162,He cannot do 7.Asking if you can do 8pm?,He cannot do 7.Asking if you can do 8pm?
1789,What are you ordering for dinner? Might need t...,What are you ordering for dinner? Might need t...
1794,Hello Dear=2CMy name is Mr Usman Lama=2C I am ...,Hello Dear=2CMy name is Mr Usman Lama=2C I am ...
3784,I am nearly done w my editing (along w Bill's)...,I am nearly done w my editing (along w Bill's)...


- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters

- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [8]:
def basic_clean(text):
    # Remove special characters and numbers, keep alphabetic words only.
    text = re.sub(r"[^A-Za-z\s]", " ", text)
    text = re.sub(r"\d+", " ", text)

    # Remove one-character tokens (including a leading single character).
    text = re.sub(r"\b[A-Za-z]\b", " ", text)

    # Remove a prefixed 'b' that may appear in byte-string representations.
    text = re.sub(r"^\s*b\s+", " ", text, flags=re.IGNORECASE)

    # Convert to lowercase and collapse repeated whitespace.
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

for frame in (data_train, data_val):
    frame["preprocessed_text"] = frame["preprocessed_text"].apply(basic_clean)

data_train["preprocessed_text"].head()


,preprocessed_text
1416,dear friend please reply to my private mail re...
4162,he cannot do asking if you can do pm
1789,what are you ordering for dinner might need to...
1794,hello dear cmy name is mr usman lama am former...
3784,am nearly done my editing along bill pls send ...


## Now let's work on removing stopwords
Remove the stopwords.

In [9]:
stop_words = set(ENGLISH_STOP_WORDS)

def remove_stopwords(text):
    return " ".join(word for word in text.split() if word not in stop_words)

for frame in (data_train, data_val):
    frame["preprocessed_text"] = frame["preprocessed_text"].apply(remove_stopwords)

data_train["preprocessed_text"].head()


,preprocessed_text
1416,dear friend reply private mail remit dept lati...
4162,asking pm
1789,ordering dinner need order
1794,hello dear cmy mr usman lama military intellig...
3784,nearly editing pls send commerts just listed r...


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [10]:
# Use WordNet lemmatization when the corpus is installed. In offline environments
# where WordNet is unavailable, fall back to Snowball stemming so the notebook
# still runs from start to finish.
def normalize_words(text):
    words = text.split()
    try:
        return " ".join(wordnet_lemmatizer.lemmatize(word) for word in words)
    except LookupError:
        return " ".join(snowball.stem(word) for word in words)

for frame in (data_train, data_val):
    frame["preprocessed_text"] = frame["preprocessed_text"].apply(normalize_words)

data_train["preprocessed_text"].head()


,preprocessed_text
1416,dear friend repli privat mail remit dept latin...
4162,ask pm
1789,order dinner need order
1794,hello dear cmi mr usman lama militari intellig...
3784,near edit pls send commert just list review


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [11]:
# Fit CountVectorizer on the training partition only to avoid validation leakage.
eda_vectorizer = CountVectorizer()
X_eda = eda_vectorizer.fit_transform(data_train["preprocessed_text"])
feature_names = np.array(eda_vectorizer.get_feature_names_out())

def top_words_for_label(label, n=10):
    counts = np.asarray(X_eda[data_train["label"].to_numpy() == label].sum(axis=0)).ravel()
    top_idx = counts.argsort()[::-1][:n]
    return pd.DataFrame({"word": feature_names[top_idx], "count": counts[top_idx]})

top_ham = top_words_for_label(0)
top_spam = top_words_for_label(1)

print("Top 10 HAM words")
display(top_ham)
print("Top 10 SPAM words")
display(top_spam)


Top 10 HAM words


,word,count
0,state,930
1,pm,774
2,secretari,506
3,time,479
4,offic,468
5,obama,458
6,depart,456
7,work,444
8,presid,426
9,said,423


Top 10 SPAM words


,word,count
0,money,4510
1,bank,4287
2,account,4103
3,fund,3567
4,nbsp,3202
5,transact,2343
6,transfer,2302
7,foreign,2252
8,countri,2239
9,busi,2209


## Extra features

In [12]:
# Add three non-negative extra indicators: money symbols/terms, suspicious words,
# and preprocessed message length. These can be concatenated with sparse text features.
money_pattern = r"(?:\beuro\b|\bdollar\b|\bpound\b|€|\$)"
suspicious_pattern = r"(?:\bfree\b|\bcheap\b|\bsex\b|\bmoney\b|\baccount\b|\bbank\b|\bfund\b|\btransfer\b|\btransaction\b|\bwin\b|\bdeposit\b|\bpassword\b)"

for frame in (data_train, data_val):
    frame["money_mark"] = frame["preprocessed_text"].str.contains(money_pattern, regex=True, case=False).astype(int)
    frame["suspicious_words"] = frame["preprocessed_text"].str.contains(suspicious_pattern, regex=True, case=False).astype(int)
    frame["text_len"] = frame["preprocessed_text"].str.len()

data_train.head()


,text,label,preprocessed_text,money_mark,suspicious_words,text_len
1416,"DEAR FRIEND,PLEASE, REPLY TO MY PRIVATE E-MAIL...",1,dear friend repli privat mail remit dept latin...,1,1,1552
4162,He cannot do 7.Asking if you can do 8pm?,0,ask pm,0,0,6
1789,What are you ordering for dinner? Might need t...,0,order dinner need order,0,0,23
1794,Hello Dear=2CMy name is Mr Usman Lama=2C I am ...,1,hello dear cmi mr usman lama militari intellig...,0,1,526
3784,I am nearly done w my editing (along w Bill's)...,0,near edit pls send commert just list review,0,0,43


## How would work the Bag of Words with Count Vectorizer concept?

In [13]:
# Bag of Words represents each document by word occurrence counts.
count_vectorizer = CountVectorizer()
X_train_bow = count_vectorizer.fit_transform(data_train["preprocessed_text"])
X_val_bow = count_vectorizer.transform(data_val["preprocessed_text"])

print("BoW training matrix shape:", X_train_bow.shape)
print("BoW validation matrix shape:", X_val_bow.shape)
print("Vocabulary size:", len(count_vectorizer.vocabulary_))


BoW training matrix shape: (4771, 74163)
BoW validation matrix shape: (1193, 74163)
Vocabulary size: 74163


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [14]:
# TF-IDF gives more weight to terms that are informative for a document and less
# weight to terms that are common across many documents.
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train["preprocessed_text"])
X_val_tfidf = tfidf_vectorizer.transform(data_val["preprocessed_text"])

print("TF-IDF training matrix shape:", X_train_tfidf.shape)
print("TF-IDF validation matrix shape:", X_val_tfidf.shape)


TF-IDF training matrix shape: (4771, 74163)
TF-IDF validation matrix shape: (1193, 74163)


## And the Train a Classifier?

In [15]:
# Train the required Multinomial Naive Bayes classifier using TF-IDF features.
y_train = data_train["label"].to_numpy()
y_val = data_val["label"].to_numpy()

clf = MultinomialNB()
clf.fit(X_train_tfidf, y_train)
y_pred = clf.predict(X_val_tfidf)

print("Accuracy:", round(accuracy_score(y_val, y_pred), 4))
print("F1 score (spam):", round(f1_score(y_val, y_pred), 4))
print("Confusion matrix:")
print(confusion_matrix(y_val, y_pred))
print("Classification report:")
print(classification_report(y_val, y_pred, target_names=["HAM", "SPAM"]))


Accuracy: 0.9581
F1 score (spam): 0.9534
Confusion matrix:
[[631  46]
 [  4 512]]
Classification report:
              precision    recall  f1-score   support

         HAM       0.99      0.93      0.96       677
        SPAM       0.92      0.99      0.95       516

    accuracy                           0.96      1193
   macro avg       0.96      0.96      0.96      1193
weighted avg       0.96      0.96      0.96      1193



### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [16]:
# Compare the four feature sets requested in the extra task. The classifier is
# MultinomialNB() with default parameters in every experiment.
extra_cols = ["money_mark", "suspicious_words", "text_len"]
train_extra = csr_matrix(data_train[extra_cols].to_numpy(dtype=float))
val_extra = csr_matrix(data_val[extra_cols].to_numpy(dtype=float))

feature_sets = {
    "Bag of Words only": (X_train_bow, X_val_bow),
    "TF-IDF only": (X_train_tfidf, X_val_tfidf),
    "Bag of Words + extra flags": (hstack([X_train_bow, train_extra]).tocsr(), hstack([X_val_bow, val_extra]).tocsr()),
    "TF-IDF + extra flags": (hstack([X_train_tfidf, train_extra]).tocsr(), hstack([X_val_tfidf, val_extra]).tocsr()),
}

results = []
for name, (Xtr, Xva) in feature_sets.items():
    model = MultinomialNB()
    model.fit(Xtr, y_train)
    pred = model.predict(Xva)
    results.append({
        "features": name,
        "accuracy": accuracy_score(y_val, pred),
        "f1_spam": f1_score(y_val, pred),
    })

results_df = pd.DataFrame(results).sort_values(["f1_spam", "accuracy"], ascending=False).reset_index(drop=True)
display(results_df)

best_features = results_df.loc[0, "features"]
print(f"Best validation feature set: {best_features}")

# Optional Kaggle-style predictions if the unlabeled test file is available.
test_candidates = [Path("data/kg_test.csv"), Path("kg_test.csv"), Path("kg_test(2).csv")]
test_path = next((p for p in test_candidates if p.exists()), None)

if test_path is not None:
    test_data = pd.read_csv(test_path, encoding="latin-1")
    test_data["text"] = test_data["text"].fillna("").astype(str)
    test_data["preprocessed_text"] = test_data["text"].apply(remove_html).apply(basic_clean).apply(remove_stopwords).apply(normalize_words)
    test_data["money_mark"] = test_data["preprocessed_text"].str.contains(money_pattern, regex=True, case=False).astype(int)
    test_data["suspicious_words"] = test_data["preprocessed_text"].str.contains(suspicious_pattern, regex=True, case=False).astype(int)
    test_data["text_len"] = test_data["preprocessed_text"].str.len()

    # Refit the selected representation on ALL labeled data for final predictions.
    all_text = pd.concat([data_train["preprocessed_text"], data_val["preprocessed_text"]], ignore_index=True)
    all_y = np.concatenate([y_train, y_val])
    all_extra = csr_matrix(pd.concat([data_train[extra_cols], data_val[extra_cols]], ignore_index=True).to_numpy(dtype=float))
    test_extra = csr_matrix(test_data[extra_cols].to_numpy(dtype=float))

    if best_features.startswith("Bag of Words"):
        final_vectorizer = CountVectorizer()
    else:
        final_vectorizer = TfidfVectorizer()

    X_all_text = final_vectorizer.fit_transform(all_text)
    X_test_text = final_vectorizer.transform(test_data["preprocessed_text"])

    if "extra flags" in best_features:
        X_all_final = hstack([X_all_text, all_extra]).tocsr()
        X_test_final = hstack([X_test_text, test_extra]).tocsr()
    else:
        X_all_final = X_all_text
        X_test_final = X_test_text

    final_model = MultinomialNB()
    final_model.fit(X_all_final, all_y)
    test_predictions = final_model.predict(X_test_final)

    submission = pd.DataFrame({"label": test_predictions})
    submission.to_csv("submission.csv", index=False)
    print("Created submission.csv with", len(submission), "predictions.")
    display(submission.head())
else:
    print("No kg_test.csv found; validation comparison is complete.")


,features,accuracy,f1_spam
0,Bag of Words only,0.971500,0.967803
1,Bag of Words + extra flags,0.968148,0.964151
2,TF-IDF only,0.958089,0.953445
3,TF-IDF + extra flags,0.827326,0.833063


Best validation feature set: Bag of Words only
Created submission.csv with 5964 predictions.


,label
0,1
1,0
2,0
3,0
4,1
